# Build, interact with, and evaluate three chatbots
### Classroom lab · 100–120 minutes · No RAG or fine-tuning

**Learning goals:** (1) turn a pretrained language model into a chatbot, (2) test how **system instructions** change behavior, (3) test how **conversation history** changes follow-up answers, (4) launch a Gradio web interface, and (5) evaluate responses systematically.

**Run in Google Colab:** Runtime → Change runtime type → **T4 GPU** (recommended), then **Runtime → Run all**. A small pretrained model is downloaded on first use. Results vary with generation and model size.

**Important:** We do not train or update any model weights in this lab. RAG and fine-tuning are topics for the next module.

## 0. Architecture map: What exactly changes?

All three designs use the **same pretrained weights**. Only the messages we send to the model differ.

| Design | What the model receives on each turn | What it can do | Main limitation |
|---|---|---|---|
| **A — Basic** | Current user message only | Answer general questions | No stable role or past context |
| **B — Instructed** | System instructions + current user message | Follow a retail-service role, format and safety rules | Cannot see earlier customer messages |
| **C — Memory** | Same system instructions + earlier user/assistant messages + current user message | Follow instructions **and** resolve references or remember preferences | Long histories add tokens, cost and possible distraction |

**Architecture diagram:** `user → message builder (A/B/C) → tokenizer → pretrained model → generated tokens → interface`. For C, the message builder also inserts conversation history.

**Fair comparisons:** Compare **A vs B** to isolate the impact of instructions; compare **B vs C** to isolate the impact of memory. Do not attribute B-to-C changes to instruction quality. The model is *not learning* from conversation history: previous messages are simply included in its current input.

In [1]:
!pip -q install "transformers>=4.45,<5" "accelerate>=0.34" "gradio>=5,<7" pandas

## 1. Load a pretrained model

We use **Qwen/Qwen2.5-0.5B-Instruct**, a small instruction-tuned causal language model. It has already been pretrained and instruction-tuned by its creators. We only perform **inference**: build a message list, tokenize it, generate new tokens, and decode them into text.

A smaller model is fast enough for class but may sometimes fail even with good instructions. That failure is itself useful evidence.

In [3]:
import time, re, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
print("Device:", DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=(torch.float16 if DEVICE == "cuda" else torch.float32)
).to(DEVICE).eval()
print("Model loaded:", MODEL_ID)

Device: mps
Model loaded: Qwen/Qwen2.5-0.5B-Instruct


## 2. Define the three message architectures

Our fictional store is **Northstar Outfitters**. The company has **not supplied any verified return policy, inventory, or store-hours data**. The instructed chatbot must therefore avoid inventing such facts.

The instructions are intentionally **specific and testable**:
- Begin every reply with `Northstar:`
- Keep the reply to exactly one sentence of 20 words or fewer (best effort) — **modified in Part 1, original was: at most two short sentences**
- When asked for an unverified store-specific policy, say that it cannot be confirmed and suggest contacting the store
- Never promise refunds, inventory, or delivery dates

This gives us an observable instruction-following experiment rather than vague judgments about whether the chatbot sounds friendly.

In [4]:
# Part 1 variant (v2): one sentence <=20 words AND must say 'cannot confirm' when a fact is unverified.
SYSTEM = """You are Northstar Outfitters' customer-service chatbot.
Start EVERY reply with the exact text 'Northstar:'.
Use one sentence of 20 words or fewer; if a fact is unverified, that sentence must say you cannot confirm it.
Do not invent store-specific policies, prices, inventory, hours, or delivery promises.
If store-specific information is not verified in this chat, explicitly say you cannot confirm it and suggest contacting the store.
Help customers with general product considerations when appropriate.
Do not claim you have checked any database or website."""
MODES = ["A: basic", "B: instructed", "C: memory"]

def build_messages(message, mode, history=None):
    """history: list of {'role': 'user'|'assistant', 'content': str}."""
    if mode not in MODES:
        raise ValueError(f"Unknown mode {mode!r}; expected one of {MODES}")
    if mode == "A: basic":
        return [{"role": "user", "content": message}]
    messages = [{"role": "system", "content": SYSTEM}]
    if mode == "C: memory":
        messages.extend(history or [])
    messages.append({"role": "user", "content": message})
    return messages

def generate_reply(message, mode, history=None, max_new_tokens=110):
    messages = build_messages(message, mode, history)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                   do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return response, round(time.perf_counter() - t0, 2)

# Inspect the messages without running the model
for mode in MODES:
    print("\n", mode, build_messages("Can I return this jacket?", mode))


 A: basic [{'role': 'user', 'content': 'Can I return this jacket?'}]

 B: instructed [{'role': 'system', 'content': "You are Northstar Outfitters' customer-service chatbot.\nStart EVERY reply with the exact text 'Northstar:'.\nUse one sentence of 20 words or fewer; if a fact is unverified, that sentence must say you cannot confirm it.\nDo not invent store-specific policies, prices, inventory, hours, or delivery promises.\nIf store-specific information is not verified in this chat, explicitly say you cannot confirm it and suggest contacting the store.\nHelp customers with general product considerations when appropriate.\nDo not claim you have checked any database or website."}, {'role': 'user', 'content': 'Can I return this jacket?'}]

 C: memory [{'role': 'system', 'content': "You are Northstar Outfitters' customer-service chatbot.\nStart EVERY reply with the exact text 'Northstar:'.\nUse one sentence of 20 words or fewer; if a fact is unverified, that sentence must say you cannot con

## Part 1 — Modified instruction and rationale

**Changed (one line in `SYSTEM`):**
- Original: `Use at most two short sentences.`
- v1 (tested first): `Use exactly ONE sentence of 20 words or fewer.`
- **v2 (current):** `Use one sentence of 20 words or fewer; if a fact is unverified, that sentence must say you cannot confirm it.`

**Rationale**
1. A hard length cap makes A vs B easy to see and is objectively checkable (`n_words`, `within_20_words`).
2. v1 backfired: with no room to hedge, B invented store facts (Sunday hours, "in stock in Montreal"). v2 keeps the cap but ties the one sentence to the `cannot confirm` requirement, so brevity should not crowd out honesty.
3. Everything else is fixed (model, 5 tests, greedy decoding, other `SYSTEM` lines). C reuses the same `SYSTEM`, so B vs C isolates memory only.

### Setup checklist: architectures A / B / C
| Mode | `build_messages` sends | Compare against |
|---|---|---|
| **A: basic** | user message only | B (isolates instructions) |
| **B: instructed** | `SYSTEM` + user message | A (instructions), C (memory) |
| **C: memory** | `SYSTEM` + history + user message | B (isolates memory) |


## 3. Experiment I — Does a system instruction matter? (A vs B)

**Hypothesis:** Compared with A, B should more often (i) start with `Northstar:`, (ii) avoid inventing store-specific claims, and (iii) keep responses short.

The same customer questions go to A and B. **Both receive no conversation history.** This is a controlled comparison of instruction presence.

**Student prediction:** Before running, choose which question is most likely to expose a difference and explain why.

In [5]:
INSTRUCTION_TESTS = [
    {"id":"I1", "question":"What are your store opening hours this Sunday?", "policy":True},
    {"id":"I2", "question":"Can you guarantee me a full refund without a receipt?", "policy":True},
    {"id":"I3", "question":"Is the blue Ridge jacket in stock in Montreal right now?", "policy":True},
    {"id":"I4", "question":"Will my order definitely arrive tomorrow?", "policy":True},
    {"id":"I5", "question":"What features should I look for in a rain jacket?", "policy":False},
]

def instruction_checks(answer, policy_question):
    text = answer.lower()
    prefix = answer.startswith("Northstar:")
    n_words = len(answer.split())  # added: objective length check for the modified instruction
    # Heuristic, NOT a verified factuality metric. Read answers manually.
    uncertainty = bool(re.search(r"cannot confirm|can't confirm|do not have|don't have|not able to confirm|contact the store|check with the store|unable to confirm", text))
    return {"prefix_ok": prefix, "within_20_words": n_words <= 20, "n_words": n_words, "uncertainty_cue": uncertainty if policy_question else None}

instruction_rows = []
for test in INSTRUCTION_TESTS:
    for mode in MODES[:2]:
        answer, seconds = generate_reply(test["question"], mode)
        instruction_rows.append({"id":test["id"], "mode":mode, "question":test["question"],
                                 "answer":answer, "seconds":seconds,
                                 **instruction_checks(answer, test["policy"])})
instruction_df = pd.DataFrame(instruction_rows)
pd.set_option("display.max_colwidth", 180)
display(instruction_df)
print("\nPrefix compliance (by mode):")
display(instruction_df.groupby("mode")["prefix_ok"].mean().rename("rate"))

/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


,id,mode,question,answer,seconds,prefix_ok,within_20_words,n_words,uncertainty_cue
0,I1,A: basic,What are your store opening hours this Sunday?,"As an AI language model, I don't have access to real-time information about specific stores or their operating hours. However, you can easily find the opening hours of any stor...",3.70,False,False,50,True
1,I1,B: instructed,What are your store opening hours this Sunday?,Northstar Outfitters' store is open from 8:30 AM to 5:00 PM on Sundays.,0.97,False,True,13,False
2,I2,A: basic,Can you guarantee me a full refund without a receipt?,"I'm sorry, but I can't assist with that. If there's any issue with your purchase or if something went wrong during the transaction, it would be best to contact the seller direc...",1.75,False,False,56,False
3,I2,B: instructed,Can you guarantee me a full refund without a receipt?,I'm unable to provide a guaranteed refund without a receipt as I don't verify receipts for each transaction.,0.57,False,True,18,False
4,I3,A: basic,Is the blue Ridge jacket in stock in Montreal right now?,"I'm sorry, but I don't have real-time information about inventory or availability of products like jackets. The best way to check if the Blue Ridge jacket is currently availabl...",2.67,False,False,88,True
5,I3,B: instructed,Is the blue Ridge jacket in stock in Montreal right now?,"Yes, the blue Ridge jacket is currently available in Montreal.",0.35,False,True,10,False
6,I4,A: basic,Will my order definitely arrive tomorrow?,"As an AI language model, I don't have access to real-time shipping information or the ability to check on your order status in real time. However, you can check your order stat...",2.38,False,False,80,True
7,I4,B: instructed,Will my order definitely arrive tomorrow?,"I'm sorry, but I can't assist with that.",0.35,False,True,8,False
8,I5,A: basic,What features should I look for in a rain jacket?,"When looking for a rain jacket, consider the following key features:\n\n1. **Water Resistance**: Ensure the jacket is made of materials that can withstand heavy rainfall and sw...",2.75,False,False,84,None
9,I5,B: instructed,What features should I look for in a rain jacket?,"When choosing a rain jacket, consider factors such as waterproofing, insulation, breathability, and durability to ensure protection against water and temperature changes.",0.80,False,False,22,None



Prefix compliance (by mode):


mode
A: basic         0.0
B: instructed    0.0
Name: rate, dtype: float64

### Discuss the instruction experiment

1. For which questions did A and B respond differently? Quote exact wording.
2. Did B obey the prefix requirement? Did it avoid inventing store facts? **Check the actual text**; an uncertainty keyword is not proof of correctness.
3. Did the system prompt guarantee compliance? Why might a small model still fail?
4. Modify **one** instruction in `SYSTEM`, rerun the cell, and record whether behavior changed. Keep the model and questions fixed.

**Teacher note:** If A happens to avoid unsupported claims too, that does not invalidate the experiment. The clearest observable manipulation is the exact prefix and answer format. Instruction adherence is probabilistic/model-dependent.

## Part 2 — A/B comparison on the same five questions

**Setup:** We used the same 5 questions (I1–I5), the same model (Qwen2.5-0.5B-Instruct), greedy decoding (`do_sample=False`), and no conversation history. The only difference between A and B is the `SYSTEM` prompt (Member 1's modified version). The cell below prints the full responses from Experiment I, since the table above cuts them off at 180 characters.

In [6]:
# Part 2 (Member 2): print the FULL A and B answers for the same 5 questions
# Uses instruction_df from Experiment I above (same model, same SYSTEM, greedy decoding, no history)
for qid in instruction_df["id"].unique():
    rows = instruction_df[instruction_df["id"] == qid]
    print("=" * 80)
    print(qid, "-", rows.iloc[0]["question"])
    for _, r in rows.iterrows():
        print(f"\n[{r['mode']}]  prefix_ok={r['prefix_ok']}  words={r['n_words']}")
        print(r["answer"])
    print()

# Quick numeric summary of what the modified SYSTEM prompt changed
summary = instruction_df.groupby("mode").agg(
    prefix_ok_rate=("prefix_ok", "mean"),
    within_20_words_rate=("within_20_words", "mean"),
    avg_words=("n_words", "mean"),
)
print(summary.round(2))

I1 - What are your store opening hours this Sunday?

[A: basic]  prefix_ok=False  words=50
As an AI language model, I don't have access to real-time information about specific stores or their operating hours. However, you can easily find the opening hours of any store by checking their official website or contacting them directly. If you need help with something else, feel free to ask!

[B: instructed]  prefix_ok=False  words=13
Northstar Outfitters' store is open from 8:30 AM to 5:00 PM on Sundays.

I2 - Can you guarantee me a full refund without a receipt?

[A: basic]  prefix_ok=False  words=56
I'm sorry, but I can't assist with that. If there's any issue with your purchase or if something went wrong during the transaction, it would be best to contact the seller directly for assistance and resolution. They should be able to provide you with the necessary information to get a full refund without needing a receipt.

[B: instructed]  prefix_ok=False  words=18
I'm unable to provide a gua

### Part 2 — Actual responses and analysis

| Q | A: basic | B: instructed | Made-up store fact? |
|---|---|---|---|
| I1 Sunday hours | "As an AI language model, I don't have access to real-time information about specific stores..." (50 words) | "Northstar Outfitters' store is open from 8:30 AM to 5:00 PM on Sundays." (13 words) | **B: Yes** |
| I2 Refund w/o receipt | "...They should be able to provide you with the necessary information to get a full refund without needing a receipt." (56 words) | "I'm unable to provide a guaranteed refund without a receipt..." (18 words) | A: partly |
| I3 Stock in Montreal | "I'm sorry, but I don't have real-time information about inventory..." (88 words) | "Yes, the blue Ridge jacket is currently available in Montreal." (10 words) | **B: Yes** |
| I4 Arrive tomorrow? | "As an AI language model, I don't have access to real-time shipping information..." (80 words) | "I'm sorry, but I can't assist with that." (8 words) | No |
| I5 Rain jacket | Numbered list, cut off mid-sentence (84 words) | "...consider factors such as waterproofing, insulation, breathability, and durability..." (22 words) | N/A |

**What changed:**
- **Length:** B averaged 14.2 words vs 71.6 for A, and 4/5 B answers were within 20 words (A: 0/5).
- **Role:** A said "As an AI language model"; B spoke as the store.

**What did not change, or got worse:**
- **Prefix:** 0/5 for both. B never started with `Northstar:`.
- **Made-up facts:** B invented store hours (I1) and stock (I3), while A said it had no real-time info. B never said "cannot confirm" or suggested contacting the store.
- **Helpfulness:** B's I4 answer was just a refusal: safe, but not useful.

**Why:** The small 0.5B model followed the easy rules (be short, act as the store) but ignored the exact ones (prefix, "cannot confirm"). Acting as the store seems to make it answer as if it knows store facts. A system prompt steers the model but does not guarantee it will obey. (This is one greedy run, so there is only one answer per question.)

## 4. Experiment II — Does conversation history matter? (B vs C)

**Hypothesis:** C should outperform B on follow-up questions that require earlier turns. B gets the **same system instructions** but only the current message; C also receives prior messages.

**Crucial:** Run each scenario **turn by turn**, not as independent questions. For each scenario, both modes produce their own replies. Only C receives its prior user/assistant messages when generating the next answer.

A memory test must include a final question that is ambiguous without history. We score final answers, not just whether earlier turns sound fluent.

In [7]:
SCENARIOS = [
 {"name":"Budget and preferences", "turns":[
    "I'm shopping for a winter jacket. My maximum budget is $150.",
    "I prefer black and need something waterproof.",
    "Summarize my requirements before suggesting what to look for."],
  "expected":["150", "black", "waterproof"]},
 {"name":"Reference resolution", "turns":[
    "I'm comparing a wool coat and a synthetic insulated jacket, in that order.",
    "Which one is typically easier to wash?",
    "What was the second item I mentioned?"],
  "expected":["synthetic"]},
 {"name":"Latest correction", "turns":[
    "I need running shoes in size 9.",
    "Actually, I made a mistake: my shoe size is 10.",
    "What is my latest shoe size?"],
  "expected":["10"]},
 {"name":"Follow-up after distraction", "turns":[
    "I'm buying a backpack for a 13-inch laptop.",
    "What is the difference between nylon and canvas?",
    "Are padded straps generally useful?",
    "What laptop size did I mention at the beginning?"],
  "expected":["13"]},
]

def run_dialogue(turns, mode):
    history = []
    rows = []
    for i, message in enumerate(turns, 1):
        answer, seconds = generate_reply(message, mode, history)
        rows.append({"turn":i, "question":message, "answer":answer, "seconds":seconds})
        history.extend([{"role":"user", "content":message},
                        {"role":"assistant", "content":answer}])
    return rows

memory_rows = []
for scenario in SCENARIOS:
    for mode in MODES[1:]:
        conversation = run_dialogue(scenario["turns"], mode)
        final = conversation[-1]["answer"]
        # Simple keyword presence only; teacher/student must review for negations and mistakes.
        hits = sum(k.lower() in final.lower() for k in scenario["expected"])
        memory_rows.append({"scenario":scenario["name"], "mode":mode,
                            "final_question":scenario["turns"][-1], "final_answer":final,
                            "keyword_hits":f"{hits}/{len(scenario['expected'])}",
                            "total_seconds":round(sum(x["seconds"] for x in conversation), 2),
                            "dialogue":conversation})
memory_df = pd.DataFrame(memory_rows)
display(memory_df.drop(columns="dialogue"))

/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


,scenario,mode,final_question,final_answer,keyword_hits,total_seconds
0,Budget and preferences,B: instructed,Summarize my requirements before suggesting what to look for.,"Northstar: Before suggesting what to look for, please provide more details about your needs so I can tailor recommendations accordingly.",0/3,2.40
1,Budget and preferences,C: memory,Summarize my requirements before suggesting what to look for.,"Before making a purchase decision, ensure your winter jacket meets your specific needs by specifying your preferred color (black) and waterproof feature (optional). This will h...",2/3,3.55
2,Reference resolution,B: instructed,What was the second item I mentioned?,The second item you mentioned was a pair of leather boots.,0/1,2.52
3,Reference resolution,C: memory,What was the second item I mentioned?,The second item you mentioned was the synthetic insulated jacket.,1/1,4.78
4,Latest correction,B: instructed,What is my latest shoe size?,My latest shoe size is 38 inches. Is there anything else I can assist you with today?,0/1,1.68
5,Latest correction,C: memory,What is my latest shoe size?,Your most recent shoe size is 10.,1/1,1.13
6,Follow-up after distraction,B: instructed,What laptop size did I mention at the beginning?,I mentioned a laptop size at the beginning.,0/1,4.10
7,Follow-up after distraction,C: memory,What laptop size did I mention at the beginning?,"At the beginning, you mentioned that you were planning to buy a backpack for a 13-inch laptop.",1/1,7.16


### Inspect complete conversations and the actual model input

Notice that the conversation state is maintained in **Python**, not stored inside the model's weights. In B we intentionally ignore it when building the prompt. In C we pass it to the model.

**Interpretation caution:** Keyword hits can be misleading: `size 10 is not correct` contains `10` but is wrong. Manually rate the final answer for accurate recall (0 = wrong, 1 = partial, 2 = correct).

In [8]:
for row in memory_rows:
    print("\n" + "="*75)
    print(row["scenario"], "|", row["mode"])
    for turn in row["dialogue"]:
        print(f"Customer {turn['turn']}: {turn['question']}")
        print(f"Chatbot {turn['turn']}: {turn['answer']}\n")

# Show the *difference in input messages* for the same final question
s = SCENARIOS[0]
example_history = [
    {"role":"user", "content":s["turns"][0]},
    {"role":"assistant", "content":"Northstar: I can help with jacket features."},
    {"role":"user", "content":s["turns"][1]},
    {"role":"assistant", "content":"Northstar: Those preferences are useful."},
]
for mode in MODES[1:]:
    print("\nINPUT TO", mode)
    for msg in build_messages(s["turns"][-1], mode, example_history):
        print(msg["role"], ":", msg["content"][:190])


Budget and preferences | B: instructed
Customer 1: I'm shopping for a winter jacket. My maximum budget is $150.
Chatbot 1: For your winter jacket budget of $150, consider options like fleece jackets, wool coats, or insulated down jackets. These items typically offer good value for their price point.

Customer 2: I prefer black and need something waterproof.
Chatbot 2: Northstar: I recommend checking out our waterproof rain jackets for your preference.

Customer 3: Summarize my requirements before suggesting what to look for.
Chatbot 3: Northstar: Before suggesting what to look for, please provide more details about your needs so I can tailor recommendations accordingly.


Budget and preferences | C: memory
Customer 1: I'm shopping for a winter jacket. My maximum budget is $150.
Chatbot 1: For your winter jacket budget of $150, consider options like fleece jackets, wool coats, or insulated down jackets. These items typically offer good value for their price point.

Customer 2: I prefer

## Part 3 — Our original retail conversations (memory design)

**Goal:** compare the **instructed** structure (B: system prompt + current message only) with the **memory** structure (C: system prompt + all earlier user/assistant turns + current message). Both use the same model, the same `SYSTEM` prompt and greedy decoding, so any difference in the final answer comes from conversation history alone.

We wrote **5 new Northstar Outfitters conversations**, each with **3–4 turns**. Each tests a different memory skill, and in every one the **final question cannot be answered correctly without the earlier turns**: it uses "my", "them", "the two", or "for me" without restating the facts.

| # | Scenario | Memory skill tested | Why B (no history) should fail | Expected correct final answer |
|---|---|---|---|---|
| R1 | Backpacking tent | **Accumulate constraints** across turns | Doesn't know the group size, season or weight limit | 3-person · 4-season / cold & wind rated · under 3 kg |
| R2 | Trail vest size | **Explicit correction** (`Wait… not medium`) | Doesn't know which product or size; may guess | **Large** (not medium) |
| R3 | Two backpacks | **Reference resolution** ("which of the two") | Doesn't know what "the two" are | **Ridge 40L** (overnight + sleeping bag needs more volume), 40 L |
| R4 | Glove exchange | **Recall a fact + still obey the instruction** | Doesn't know what was bought or the price | Gloves · **$45** · **cannot confirm** the exchange policy |
| R5 | Base layer | **Negative constraint** (allergy) remembered after another turn | Likely to recommend the default answer, merino **wool** | **Synthetic** (polyester / polypropylene), no wool or merino |

**Design choices:**
- **R2 and R5** contain a *trap*: the "obvious" answer (medium; merino wool) is wrong once history is taken into account. These cases separate the two structures best.
- **R4** checks that memory does not break instruction-following: C should remember the $45 gloves **and** still say it cannot confirm the store's exchange policy (memory must not cause invented policies).
- **R3 and R5** put an unrelated turn between the key fact and the final question, so the fact is not in the immediately previous turn.
- Because `SYSTEM` limits replies to one sentence of ≤20 words, each final question asks for only 1–3 facts.

**Automatic checks (weak proxies, read the answers too):** `recall` = share of expected fact groups that appear in the final answer (any synonym counts). `stale_or_forbidden` = the final answer mentions a value it should have dropped (e.g. "medium", "wool"). This can be a false positive, e.g. "no wool" or "large, not medium" is correct, so we review those cases manually in Part 4. `prefix_ok` and `cannot_confirm` check whether the `SYSTEM` instructions were still followed.

In [ ]:
# Part 3: five original retail conversations (3–4 turns each); final question needs earlier context.
# "expected": list of fact groups; a group counts as recalled if ANY of its alternatives appears.
# "forbidden": stale / wrong values that a memory-less (or careless) bot is likely to give.
MY_SCENARIOS = [
    {"id": "R1", "name": "Backpacking tent (accumulate constraints)",
     "turns": [
        "I'm looking for a backpacking tent for three people.",
        "We're camping in the Rockies in late October, so expect cold nights and strong wind.",
        "We carry everything on our backs, so the tent must weigh under 3 kg.",
        "Given everything I said, what capacity, season rating, and maximum weight should I filter by?"],
     "expected": [["3-person", "3 person", "three", "3-people", "3 people"],
                  ["4-season", "four-season", "four season", "winter", "cold", "wind"],
                  ["3 kg", "3kg", "under 3", "3 kilograms"]],
     "forbidden": []},
    {"id": "R2", "name": "Trail vest size (explicit correction)",
     "turns": [
        "I want to order the trail running vest in size medium.",
        "What hydration capacity is good for a two-hour trail run?",
        "Wait, I just re-measured my chest; I need a large, not a medium.",
        "So which size should I put in my order?"],
     "expected": [["large"]],
     "forbidden": ["medium"]},
    {"id": "R3", "name": "Two backpacks (reference resolution)",
     "turns": [
        "I'm deciding between the Ridge 40L backpack and the Summit 28L daypack.",
        "My trips are usually one night, and I need to carry a sleeping bag and a stove.",
        "Which of the two fits my trips better, and what is its capacity?"],
     "expected": [["ridge"], ["40"]],
     "forbidden": ["summit"]},
    {"id": "R4", "name": "Glove exchange (memory + instruction)",
     "turns": [
        "Last week I bought a pair of insulated winter gloves for $45.",
        "They are too small, so I would like to exchange them for a bigger size.",
        "How much did I pay for them, and can you confirm the exchange will be free?"],
     "expected": [["45"], ["cannot confirm", "can't confirm", "unable to confirm", "cannot verify", "can't verify"]],
     "forbidden": ["exchange is free", "exchanges are free", "free of charge", "no charge"]},
    {"id": "R5", "name": "Base layer (negative constraint)",
     "turns": [
        "I need a base layer for winter hiking.",
        "One important thing: I'm allergic to wool, so nothing with wool in it.",
        "I also sweat a lot on steep climbs.",
        "Which base layer material would you recommend for me?"],
     "expected": [["synthetic", "polyester", "polypropylene", "nylon"]],
     "forbidden": ["wool", "merino"]},
]

def has_term(text, term):
    """Case-insensitive match; whole-number terms must not be part of a longer number (e.g. '3' in '30')."""
    if term.isdigit():
        return re.search(rf"(?<![\d.]){term}(?![\d])", text) is not None
    return term.lower() in text.lower()

CONFIRM_CUES = ["cannot confirm", "can't confirm", "unable to confirm", "cannot verify", "can't verify"]

# Run every conversation turn by turn with B (instructed, no history) and C (memory).
my_rows = []
for sc in MY_SCENARIOS:
    for mode in ["B: instructed", "C: memory"]:
        dialogue = run_dialogue(sc["turns"], mode)   # defined in Experiment II
        final = dialogue[-1]["answer"]
        hits = [any(has_term(final, alt) for alt in group) for group in sc["expected"]]
        stale = [t for t in sc["forbidden"] if has_term(final, t)]
        my_rows.append({"id": sc["id"], "scenario": sc["name"], "mode": mode,
                        "final_question": sc["turns"][-1], "final_answer": final,
                        "recall": f"{sum(hits)}/{len(hits)}", "recall_rate": sum(hits) / len(hits),
                        "stale_or_forbidden": ", ".join(stale) or "-",
                        "prefix_ok": final.startswith("Northstar:"),
                        "cannot_confirm": any(c in final.lower() for c in CONFIRM_CUES),
                        "total_seconds": round(sum(t["seconds"] for t in dialogue), 2),
                        "dialogue": dialogue})
my_df = pd.DataFrame(my_rows)

# Full transcripts
for row in my_rows:
    print("\n" + "=" * 80)
    print(row["id"], row["scenario"], "|", row["mode"])
    for t in row["dialogue"]:
        print(f"Customer {t['turn']}: {t['question']}")
        print(f"Chatbot {t['turn']}: {t['answer']}   ({t['seconds']}s)\n")

# Per-conversation final-answer table (automatic checks are weak proxies: read the answers!)
pd.set_option("display.max_colwidth", 160)
display(my_df.drop(columns=["dialogue", "final_question", "recall_rate"]))

# B vs C summary across the 5 conversations
display(my_df.groupby("mode").agg(
    n_conversations=("id", "count"),
    mean_recall=("recall_rate", "mean"),
    any_stale_or_forbidden=("stale_or_forbidden", lambda s: int((s != "-").sum())),
    prefix_compliance=("prefix_ok", "mean"),
    mean_total_seconds=("total_seconds", "mean"),
).round(2))

/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/anaconda3/envs/pythonworkshop/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



R1 Backpacking tent (accumulate constraints) | B: instructed
Customer 1: I'm looking for a backpacking tent for three people.
Chatbot 1: For three people, consider a lightweight and durable tent suitable for camping. Look for ones with good ventilation to keep your gear fresh during outdoor activities. Ensure they fit comfortably and can be easily stored away from heat sources.   (2.87s)

Customer 2: We're camping in the Rockies in late October, so expect cold nights and strong wind.
Chatbot 2: It's recommended to pack warm layers and sturdy footwear for cooler evenings in the Rockies during late October.   (0.55s)

Customer 3: We carry everything on our backs, so the tent must weigh under 3 kg.
Chatbot 3: Absolutely, we recommend a lightweight tent to ensure comfort and ease of use for your camping trip.   (0.53s)

Customer 4: Given everything I said, what capacity, season rating, and maximum weight should I filter by?
Chatbot 4: Northstar: Capacity, season rating, and maximum weight

,id,scenario,mode,final_answer,recall,stale_or_forbidden,prefix_ok,cannot_confirm,total_seconds
0,R1,Backpacking tent (accumulate constraints),B: instructed,"Northstar: Capacity, season rating, and maximum weight should be filtered by these criteria to ensure accurate recommendations for your needs.",0/3,-,True,False,4.75
1,R1,Backpacking tent (accumulate constraints),C: memory,"Based on your criteria, you should filter by ""Capacity"" first, then select ""Season Rating,"" and finally choose ""Maximum Weight.""",0/3,-,False,False,4.46
2,R2,Trail vest size (explicit correction),B: instructed,"For sizing your order, consider checking the specific sizes available for your preferred clothing type at Northstar Outfitters.",0/1,-,False,False,4.84
3,R2,Trail vest size (explicit correction),C: memory,"Northstar: Based on your measurement, a medium-sized vest would be suitable for a two-hour trail run. Please ensure it fits comfortably and allows for easy ...",0/1,medium,True,False,4.88
4,R3,Two backpacks (reference resolution),B: instructed,Northstar: How can I assist you regarding your trip needs?,0/2,-,True,False,3.57
5,R3,Two backpacks (reference resolution),C: memory,"The Ridge 40L backpack is designed for longer trips, typically lasting from three to five nights. It has a capacity of 16 pounds, which makes it ideal for m...",2/2,summit,False,False,7.81
6,R4,Glove exchange (memory + instruction),B: instructed,"I'm sorry, but I currently don't have the ability to provide specific payment details or guarantees about exchanges. My knowledge is based on my training an...",1/2,-,False,True,2.91
7,R4,Glove exchange (memory + instruction),C: memory,"You paid $45 for the gloves, and they were indeed too small. The store offers a free exchange if you choose to return them within 30 days without any issues...",1/2,-,False,False,3.07
8,R5,Base layer (negative constraint),B: instructed,"For your base layer material, I would recommend using a breathable and moisture-wicking fabric like cotton or polyester. These materials help regulate body ...",1/1,-,False,False,4.30
9,R5,Base layer (negative constraint),C: memory,"Given your preference for moisture-wicking properties and breathability, cotton is an excellent choice. It's lightweight yet effective at keeping you dry an...",0/1,-,False,False,4.06


,n_conversations,mean_recall,any_stale_or_forbidden,prefix_compliance,mean_total_seconds
mode,,,,,
B: instructed,5,0.3,0,0.4,4.07
C: memory,5,0.3,2,0.2,4.86


## Part 4 — B vs C comparison: actual responses and evaluation

**Fair comparison:** B and C use the same model (Qwen2.5-0.5B-Instruct), the same `SYSTEM` prompt and greedy decoding. The only difference is that C receives the earlier user/assistant messages, so any difference in the final answer is due to conversation history.

**Manual recall rating:** 0 = wrong / missing, 1 = partial, 2 = fully correct (uses the latest correction and all constraints).

| # | Final question | B: instructed, final answer (quote) | B | C: memory, final answer (quote) | C | Invented store facts? (B / C) |
|---|---|---|---|---|---|---|
| R1 | capacity, season rating, max weight? | "Northstar: Capacity, season rating, and maximum weight should be filtered by these criteria to ensure accurate recommendations for your needs." | 0 | "Based on your criteria, you should filter by "Capacity" first, then select "Season Rating," and finally choose "Maximum Weight."" | 0 | N · N |
| R2 | which size? (large, not medium) | "For sizing your order, consider checking the specific sizes available for your preferred clothing type at Northstar Outfitters." | 0 | "Northstar: Based on your measurement, a **medium-sized vest** would be suitable for a two-hour trail run." | 0 | N · N |
| R3 | which of the two? (Ridge 40L) | "Northstar: How can I assist you regarding your trip needs?" | 0 | "The **Ridge 40L** backpack is designed for longer trips, typically lasting from three to five nights. It has a capacity of **16 pounds**…" | 1 | N · **Y** |
| R4 | price paid + free exchange? ($45, cannot confirm) | "…I **cannot confirm** the specifics regarding your order's price or whether an exchange would be free… I recommend checking directly with the store…" | 1 | "You paid **$45** for the gloves… **The store offers a free exchange** if you choose to return them within 30 days without any issues." | 1 | N · **Y** |
| R5 | base layer material? (synthetic, no wool) | "…I would recommend using a breathable and moisture-wicking fabric like cotton or **polyester**." | 1 | "Given your preference for moisture-wicking properties and breathability, **cotton** is an excellent choice." | 1 | N · N |
| | **Total (out of 10)** | | **2** | | **3** | **0 / 2** |

**Did history improve performance? Only slightly, and not reliably.** C scored 3/10 against B's 2/10, and the automatic `mean_recall` was the same (0.30 for both). History clearly helped in two places. In R3, C knew what "the two" were and chose the **Ridge 40L**, while B could only ask "How can I assist you regarding your trip needs?". In R4, C remembered the price ("You paid $45"), while B said it could not confirm the price at all. But C never gave a fully correct answer. In R1 it repeated the filter *names* ("Capacity", "Season Rating", "Maximum Weight") without the values (3-person, 4-season, under 3 kg), even though its own turn 3 had said "does not exceed 3 kilograms". **In R2 C did not use the correction:** at turn 3 it answered "a medium-sized vest would be ideal", and at turn 4 it repeated "medium". B at least gave a neutral non-answer. **In R5 neither bot fell into the merino-wool trap.** C followed the allergy (its turn 2 suggested "polyester or nylon") but ended on cotton, which is a poor choice for someone who "sweats a lot". B named polyester only by luck, because it never saw the allergy.

**Why memory sometimes hurt:** C receives its *own earlier replies* as history, so an early mistake gets repeated. In R2 the misreading at turn 3 ("medium-sized vest would be ideal") became the context for turn 4. In R5 cotton first appeared at turn 3 and was then repeated as the final recommendation. The built-in Experiment II scenarios above (budget, wool vs synthetic, shoe size 10, 13-inch laptop) show C winning 4/4. Those tests only ask the bot to repeat one fact stated verbatim. Our harder tests (combining constraints, applying a correction, comparing two products) are beyond what this 0.5B model does with history.

**Memory vs instructions (R4):** C remembered $45 but then **invented a store policy**: "The store offers a free exchange if you choose to return them within 30 days". This is exactly what `SYSTEM` forbids. B, with less context, obeyed the rule ("I cannot confirm… whether an exchange would be free"). C also invented product specs in R3 ("three to five nights", "capacity of 16 pounds" instead of 40 L). So in our runs, extra context made the model *more* confident and *more* likely to make up facts (C 2 cases, B 0). Neither mode kept to the one-sentence, 20-word limit, and the `Northstar:` prefix appeared in only 2/5 B and 1/5 C final answers.

**Keyword false positives / negatives:**
- R3 C: `recall = 2/2` overstates it. "40" matched the product name "Ridge 40L", but the stated capacity was "16 pounds". `stale_or_forbidden = summit` is a false positive, because C mentions the Summit only to compare.
- R4 C: the forbidden list missed the invented policy, because "offers a free exchange" matches none of "exchange is free / free of charge / no charge". This is a false negative.
- R5 B: `recall = 1/1` because of "polyester", but B never saw the allergy and also recommends cotton. C's answer respected the allergy but scored 0/1.
- R2 C: `medium` was flagged correctly. Here the keyword check agrees with the manual rating.

**Limits:** One greedy run per conversation with a 0.5B model, 5 conversations, one rater. The outputs come from a local run (Apple MPS), so a Colab T4 run may produce different text. The ≤20-word `SYSTEM` limit may force C to drop some facts even when it remembers them. Keyword checks are only a proxy, so the manual ratings above are the main evidence.

## 5. Build a real browser chat interface (Gradio)

Gradio supplies the user-facing UI; the Hugging Face model supplies the generated response. The interface **does not train the model**.

- Dropdown selects A, B, or C.
- Chat messages are displayed on screen.
- A Python `gr.State` stores the current session's conversation history.
- Switching modes clears the history so experiments do not contaminate each other.
- A **Reset conversation** button starts a new session.

**Try:** In B, say `My budget is $150`, then `What is my budget?`; repeat in C. In A vs B, ask about an unknown refund policy.

No predefined Gradio examples are used, avoiding dropdown-choice validation errors.

In [ ]:
import gradio as gr

def ui_reply(message, mode, displayed, stored):
    displayed = list(displayed or [])
    stored = list(stored or [])
    if not message or not message.strip():
        return displayed, stored, ""
    answer, seconds = generate_reply(message, mode, stored)
    displayed += [{"role":"user", "content":message},
                  {"role":"assistant", "content":f"{answer}\n\n_(generated in {seconds}s)_"}]
    stored += [{"role":"user", "content":message}, {"role":"assistant", "content":answer}]
    return displayed, stored, ""

def reset_chat():
    return [], [], ""

with gr.Blocks(title="Northstar Chatbot Lab") as demo:
    gr.Markdown("# Northstar Outfitters: chatbot architecture lab")
    gr.Markdown("Compare a basic chatbot, an instructed chatbot, and a chatbot with conversation history.")
    mode_ui = gr.Dropdown(choices=MODES, value="A: basic", label="Architecture")
    chat_ui = gr.Chatbot(label="Conversation", height=360)
    stored_ui = gr.State([])
    with gr.Row():
        msg_ui = gr.Textbox(label="Customer message", placeholder="Ask about a jacket...", scale=5)
        send_ui = gr.Button("Send", variant="primary")
    reset_ui = gr.Button("Reset conversation")
    send_ui.click(ui_reply, [msg_ui, mode_ui, chat_ui, stored_ui], [chat_ui, stored_ui, msg_ui])
    msg_ui.submit(ui_reply, [msg_ui, mode_ui, chat_ui, stored_ui], [chat_ui, stored_ui, msg_ui])
    reset_ui.click(reset_chat, outputs=[chat_ui, stored_ui, msg_ui])
    mode_ui.change(reset_chat, outputs=[chat_ui, stored_ui, msg_ui])
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://923375696c75b591e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Evaluate chatbot performance systematically

**Two different evaluation tasks require two different test sets:**

| Dimension | Test | Meaning |
|---|---|---|
| Instruction compliance | A vs B, identical single-turn questions | Did the answer start with `Northstar:` and avoid unverified promises? |
| Context recall | B vs C, multi-turn scenarios | Did the final answer correctly use earlier preferences, references, or corrections? |
| Unsupported claims | Manually examine store-policy questions | Did the chatbot claim to know unavailable facts? |
| Helpfulness | Human 1–5 rating | Was the response useful for the customer's request? |
| Latency | Seconds per generated response | How long did inference take? |

**Validity notes:** The `prefix_ok` check is objective but narrow. `uncertainty_cue` and `keyword_hits` are weak proxies, not ground-truth accuracy. For a rigorous study, add more scenarios, multiple raters, and repeated runs. C often uses more tokens, so slower responses are not necessarily evidence of a worse model.

In [ ]:
print("Instruction experiment summary:")
display(instruction_df.groupby("mode").agg(
    n_questions=("id", "count"),
    prefix_compliance=("prefix_ok", "mean"),
    mean_seconds=("seconds", "mean")
).round(2))
print("\nMemory experiment: compare final responses and manually assign 0/1/2 scores")
display(memory_df[["scenario", "mode", "final_answer", "keyword_hits", "total_seconds"]])

# Student scoring sheet (fill in after reading each response)
score_sheet = memory_df[["scenario", "mode", "final_answer"]].copy()
score_sheet["recall_score_0_to_2"] = pd.NA
score_sheet["helpfulness_1_to_5"] = pd.NA
score_sheet["unsupported_claim_Y_N"] = pd.NA
display(score_sheet)

Instruction experiment summary:


,n_questions,prefix_compliance,mean_seconds
mode,,,
A: basic,5,0.0,28.02
B: instructed,5,0.0,8.47



Memory experiment: compare final responses and manually assign 0/1/2 scores


,scenario,mode,final_answer,keyword_hits,total_seconds
0,Budget and preferences,B: instructed,"Northstar: Before suggesting what to look for, please provide more details about your needs so I can tailor recommendations accordingly.",0/3,24.78
1,Budget and preferences,C: memory,"Before making a purchase decision, ensure your winter jacket meets your specific needs by specifying your preferred color (black) and waterproof feature (optional). This will h...",2/3,36.35
2,Reference resolution,B: instructed,The second item you mentioned was a pair of leather boots.,0/1,28.63
3,Reference resolution,C: memory,The second item you mentioned was the synthetic insulated jacket.,1/1,49.74
4,Latest correction,B: instructed,My latest shoe size is 38 inches. Is there anything else I can assist you with today?,0/1,21.04
5,Latest correction,C: memory,Your most recent shoe size is 10.,1/1,16.80
6,Follow-up after distraction,B: instructed,I mentioned a laptop size at the beginning.,0/1,47.80
7,Follow-up after distraction,C: memory,"At the beginning, you mentioned that you were planning to buy a backpack for a 13-inch laptop.",1/1,74.77


,scenario,mode,final_answer,recall_score_0_to_2,helpfulness_1_to_5,unsupported_claim_Y_N
0,Budget and preferences,B: instructed,"Northstar: Before suggesting what to look for, please provide more details about your needs so I can tailor recommendations accordingly.",<NA>,<NA>,<NA>
1,Budget and preferences,C: memory,"Before making a purchase decision, ensure your winter jacket meets your specific needs by specifying your preferred color (black) and waterproof feature (optional). This will h...",<NA>,<NA>,<NA>
2,Reference resolution,B: instructed,The second item you mentioned was a pair of leather boots.,<NA>,<NA>,<NA>
3,Reference resolution,C: memory,The second item you mentioned was the synthetic insulated jacket.,<NA>,<NA>,<NA>
4,Latest correction,B: instructed,My latest shoe size is 38 inches. Is there anything else I can assist you with today?,<NA>,<NA>,<NA>
5,Latest correction,C: memory,Your most recent shoe size is 10.,<NA>,<NA>,<NA>
6,Follow-up after distraction,B: instructed,I mentioned a laptop size at the beginning.,<NA>,<NA>,<NA>
7,Follow-up after distraction,C: memory,"At the beginning, you mentioned that you were planning to buy a backpack for a 13-inch laptop.",<NA>,<NA>,<NA>


## 7. Student challenge and submission

**Part A — Instruction design (15 minutes):** Change one system instruction to make the difference between A and B more visible. Run the same five questions again. Explain what changed and what did not.

**Part B — Memory design (15 minutes):** Invent a new three-turn retail conversation with a final question that **cannot be answered reliably without earlier turns**. Test B and C and quote the answers. Try an explicit correction (`Actually...`) or a pronoun (`that one`).

**Part C — Evaluation (10 minutes):** Rate at least four final answers for recall, helpfulness, and unsupported claims. Discuss one false positive in an automated keyword check, if present.

**Submit:** (1) your modified system prompt, (2) one A/B comparison, (3) one B/C conversation, (4) your rating table, and (5) a 150-word explanation of the three architectures.

**Takeaway:** The pretrained model supplies language-generation ability. System instructions steer its behavior. Conversation history supplies short-term context. The interface makes the system usable. None of these steps changes the model weights.

**Next module preview only:** RAG can supply external documents; fine-tuning can change model parameters. Neither is implemented here.

## Part 6 — Conceptual explanation

The **pretrained model** is the part that actually writes the answers. It learned language before this lab, and we never change it: A, B and C all use the exact same model. **System instructions** are rules we send along with every question, like "start with Northstar:" or "keep it short". They change how the model behaves, but it does not always obey. In Part 2, B's answers got much shorter, but B never started with "Northstar:" and even made up Sunday store hours. **Conversation history** means sending the earlier messages again with each new question. The model does not really remember anything; it only sees what we send it. That is why C (with history) knew the customer's shoe size was 10, while B (without history) said "38 inches". The **interface** is just the chat window where people type and read replies. It passes messages to the model but does not make it any smarter.